# Sentence-T5 Embedding 提取 — Colab (E11 准备)

**目的**：用 `sentence-transformers/sentence-t5-base` 为所有 item 重新提取 embedding，对齐 TIGER 参考实现。产出给 E11 的 RQ-VAE + 下游 T5 用。

**输入**：`data/beauty_data.pkl`（已在 repo 里）

**产出**：`embedding/item_embeddings_raw_t5.npy`

**预计耗时**：T4 ~5 min，CPU ~20 min（12k items，batch=64）

**GPU 要求**：任意 GPU 都行（sentence-t5-base 只有 ~220 MB），不需要 A100。可以用空闲的 T4 session。

---

In [ ]:
# Cell 1：GPU 检查（非必需，CPU 也能跑）
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2：安装依赖
!pip install sentence-transformers -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：克隆仓库
!git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
%cd Generative-Sequential-Recommendation
!ls data/

In [ ]:
# Cell 4：提取 embedding
# 输出到 embedding/item_embeddings_raw_t5.npy
!python embedding/extract_embeddings_t5.py

In [ ]:
# Cell 5：下载回本地
from google.colab import files
import os
fpath = 'embedding/item_embeddings_raw_t5.npy'
if os.path.exists(fpath):
    files.download(fpath)
    print(f'✅ 下载：{fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
else:
    print('⚠️  文件不存在，Cell 4 可能失败')

---
## 下一步

下载 `item_embeddings_raw_t5.npy` 到本地后：

1. 本地修改 `embedding/rqvae.py` 顶部两行：
   ```python
   EMBEDDING_FILE = 'item_embeddings_raw_t5.npy'
   OUTPUT_TAG     = 't5_3kep'
   ```
2. commit + push
3. 开一个新 Colab session，用 `colab_rqvae.ipynb`：
   - Cell 4 上传 `item_embeddings_raw_t5.npy`（注意改文件名）
   - Cell 5 跑 rqvae，输出到 `rqvae_t5_3kep_best.pt`
   - Cell 6 生成 `semantic_ids_rqvae_t5_3kep.npy`
4. 再开一个新 Colab session 跑 T5 下游训练（E11）